# OncoVision: train all models on a free GPU

Works on **Kaggle Notebooks** (recommended: Settings → Accelerator → GPU T4, Internet → On) and on **Google Colab** (Runtime → Change runtime type → GPU).

Every dataset is downloaded automatically from public Hugging Face mirrors (all CC-BY-4.0), so no Kaggle API key is needed. Run all cells, then download `oncovision_weights.zip` from the output and unzip it into your project folder.

Rough times on a T4 GPU: brain classifier 10 min, brain segmenter 15 min, breast models 10 min.

In [ ]:
!git clone -q https://github.com/Vaishnavi220506/Medical.git oncovision || (cd oncovision && git pull -q)
%cd oncovision
!pip install -q -r requirements.txt pyarrow huggingface_hub
import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (training will be slow)")

## 1. Download datasets

In [ ]:
from huggingface_hub import snapshot_download
for repo, folder in [
    ("BJyotibrat/Masoud-Nickparvar-Brain-Tumor-MRI-Dataset", "data/brain"),     # brain tumor classification
    ("MedOtter/Figshare_Brain_Tumor", "data/figshare_raw"),                      # brain tumor masks
    ("theshoaibme/panorgan-busi-breast-ultrasound", "data/busi"),                # breast ultrasound + masks
]:
    snapshot_download(repo_id=repo, repo_type="dataset", local_dir=folder, max_workers=16)
!python scripts/prepare_figshare.py --raw data/figshare_raw --out data/figshare --negatives data/brain/Training/notumor

## 2. Brain tumor classifier (glioma / meningioma / pituitary / no tumor)

In [ ]:
!python -m oncovision.training.train_classifier --profile brain_mri --train-dir data/brain/Training --test-dir data/brain/Testing --pretrained --epochs 15 --batch-size 64 --workers 4
!python -m oncovision.training.evaluate classifier --profile brain_mri --data data/brain/Testing

## 3. Brain tumor segmenter (whole tumor, Figshare)

In [ ]:
!python -m oncovision.training.train_segmenter --profile brain_mri --format maskfolder --data data/figshare/train/images --masks data/figshare/train/masks --epochs 40 --batch-size 32 --workers 4
!python -m oncovision.training.evaluate segmenter --profile brain_mri --format maskfolder --data data/figshare/test/images --masks data/figshare/test/masks

## 4. Breast ultrasound classifier and segmenter (BUSI)

In [ ]:
!python -m oncovision.training.train_classifier --profile breast_us --train-dir data/busi --pretrained --epochs 30 --batch-size 32 --workers 4
!python -m oncovision.training.train_segmenter --profile breast_us --format busi --data data/busi --epochs 60 --batch-size 16 --workers 4
!python -m oncovision.training.evaluate segmenter --profile breast_us --format busi --data data/busi --validation-only

## 5. Optional: BraTS sub-region segmenter

Needs the BraTS 2021 data (about 13 GB, see `docs/DATASETS.md`). It replaces the Figshare segmenter with one that separates necrotic core, edema and enhancing tumor on multi-sequence NIfTI scans. Skip it if you only use JPG/PNG images.

In [ ]:
# !python -m oncovision.training.train_segmenter --profile brain_mri --format brats --data /kaggle/input/YOUR_BRATS_FOLDER --prepared-dir data/brats_slices --epochs 20 --batch-size 32 --workers 4

## 6. Package the results

In [ ]:
!zip -qr oncovision_weights.zip weights reports
print("Download oncovision_weights.zip, unzip it in your OncoVision folder, and restart the app.")